[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/24.%20The%20Static%20Truck%20Appointment%20System%20Problem/P24-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 24. The Static Truck Appointment System Problem

## Tier 3: Ant Colony Optimization

### Goal and scope

This notebook uses Ant Colony Optimization (ACO), a population-based metaheuristic, to assign truck requests to capacity-limited time slots. Each artificial ant constructs a complete assignment. Pheromone trails record useful request–slot choices, while heuristic information favors slots close to each request's preferred time. Repeated construction and pheromone updates search for a low-penalty feasible assignment.

The inputs are requests, their preferred slots and priority weights, slot capacities, and ACO parameters. The output is the best feasible assignment found during the specified search, together with convergence, pheromone, and comparison analyses. ACO is a heuristic search: it does not certify that its best solution is globally optimal, and a run may find no feasible solution even when one exists.

### Model assumptions

Each request must be assigned to exactly one time slot. A slot can serve no more requests than its stated capacity. Slots are ordered discrete time periods, so the deviation between a request's preferred and assigned times is the absolute difference between their slot indices. The model does not represent travel times, appointment durations, uncertainty, or interactions between requests beyond slot capacity.

The objective combines priority-weighted deviation from preferred slots with a small penalty for imbalance in slot loads. For requests \(i\in R\) and slots \(j\in S\), let \(p_i\) be request \(i\)'s preferred slot, \(w_i\) its priority weight, \(x_i\in S\) its assigned slot, and \(L_j\) the number of requests assigned to slot \(j\). The model is

$$
\begin{aligned}
\min_{x}\quad & \sum_{i\in R} w_i\lvert x_i-p_i\rvert
  + 0.1\left(\max_{j\in S}L_j-\min_{j\in S}L_j\right) \\
\text{subject to}\quad & L_j=\sum_{i\in R}\mathbf{1}[x_i=j], && j\in S,\\
& L_j\leq c_j, && j\in S,\\
& x_i\in S, && i\in R.
\end{aligned}
$$

Here \(c_j\) is the capacity of slot \(j\), and the indicator is 1 when its condition is true and 0 otherwise. The first term penalizes assigning a request away from its preferred time; the second discourages uneven utilization. Feasibility is determined by the capacity constraints. The coefficient 0.1 is fixed in the fitness calculation; it is not an ACO search parameter.

### How the ACO search works

Intuitively, a pheromone value \(\tau_{ij}\) is a learned preference for assigning request \(i\) to slot \(j\). The heuristic value \(\eta_{ij}=1/(1+|j-p_i|)\) is immediate information: it is larger for slots nearer the preferred time. When constructing a solution, an ant considers only slots with remaining capacity and scores each candidate in proportion to

$$
q_{ij}=\tau_{ij}^{\alpha}\eta_{ij}^{\beta}.
$$

With probability \(q_0\), the ant chooses the candidate with the largest score (exploitation); otherwise, it samples a candidate in proportion to its score (exploration). Here, \(\alpha\) controls the influence of learned pheromone, \(\beta\) controls the influence of the distance-based heuristic, and \(q_0\) controls the exploitation probability.

After all ants construct assignments, pheromone evaporates by a factor \(1-\rho\), where \(\rho\in[0,1]\) is the evaporation rate. Feasible solutions then deposit pheromone on their chosen request–slot pairs. Deposit strength decreases with fitness; the global best solution receives an additional deposit. The bounded deposit \(1/(1+\text{fitness})\) keeps updates finite even when a solution has zero penalty.

The construction is sequential: requests are processed in a randomized order, and each assignment reduces the chosen slot's remaining capacity. If no capacity remains for a request, that ant cannot produce a feasible complete assignment. A feasible ant's fitness is the objective above; infeasible assignments are not eligible to become the best solution or deposit pheromone.

### Search sequence and interpretation

The notebook defines the data structures, builds four test instances, implements the ACO class, and runs it on the extended example. It then visualizes the search, examines parameter sensitivity, and compares ACO with the EDF heuristic already included in the notebook. The tight instance has total capacity below its number of requests, so it is infeasible by construction; it provides a case for interpreting a run with no feasible solution rather than evidence of a search failure on a feasible instance.

The best fitness is the lowest objective value among feasible assignments encountered, not a proof of optimality. Pheromone levels summarize the search history and are not themselves probabilities or proof that a particular assignment is optimal. Results from the randomly generated instances are reproducible with the seed used in the data-generation cell and the ACO class.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

print("Libraries imported successfully for ACO implementation")

In [ ]:
@dataclass
class TruckRequest:
    """A truck appointment request and its preferred time slot."""

    id: int
    preferred_time: int
    weight: float
    service_type: str = "standard"


@dataclass
class TimeSlot:
    """A time slot with a maximum number of appointments."""

    id: int
    capacity: int
    current_load: int = 0


@dataclass
class ACOSolution:
    """An assignment and its objective value and feasibility status."""

    assignments: Dict[int, int]
    fitness: float
    feasible: bool


print("Data structures defined for ACO metaheuristic")

In [ ]:
def create_aco_test_instances():
    """Create the four instances used to examine ACO behavior."""
    instances = {}

    requests1 = [
        TruckRequest(1, 0, 1.0), TruckRequest(2, 0, 2.0),
        TruckRequest(3, 1, 1.0), TruckRequest(4, 1, 1.0),
        TruckRequest(5, 2, 2.0), TruckRequest(6, 3, 1.0),
        TruckRequest(7, 2, 1.0), TruckRequest(8, 1, 2.0),
    ]
    time_slots1 = [TimeSlot(i, 2) for i in range(4)]
    instances["extended"] = (requests1, time_slots1)

    requests2 = [
        TruckRequest(i, np.random.randint(0, 6), np.random.uniform(0.5, 2.0))
        for i in range(1, 16)
    ]
    time_slots2 = [TimeSlot(i, 3) for i in range(6)]
    instances["medium"] = (requests2, time_slots2)

    requests3 = [
        TruckRequest(i, np.random.randint(0, 8), np.random.uniform(0.5, 2.0))
        for i in range(1, 26)
    ]
    time_slots3 = [TimeSlot(i, 4) for i in range(8)]
    instances["large"] = (requests3, time_slots3)

    requests4 = [
        TruckRequest(i, np.random.choice([0, 1, 2]), np.random.uniform(0.5, 2.0))
        for i in range(1, 13)
    ]
    time_slots4 = [TimeSlot(0, 3), TimeSlot(1, 3), TimeSlot(2, 2)]
    instances["tight"] = (requests4, time_slots4)

    return instances


# Seeding before sampling makes the generated instances reproducible.
np.random.seed(42)
aco_instances = create_aco_test_instances()
print(f"Created {len(aco_instances)} ACO test instances:")
for name, (requests, time_slots) in aco_instances.items():
    print(f"  {name}: {len(requests)} requests, {len(time_slots)} time slots")

In [ ]:
class AntColonyOptimization:
    """ACO search for the static truck appointment assignment problem."""

    def __init__(
        self,
        requests: List[TruckRequest],
        time_slots: List[TimeSlot],
        num_ants: int = 20,
        max_iterations: int = 100,
        alpha: float = 1.0,
        beta: float = 2.0,
        rho: float = 0.1,
        q0: float = 0.9,
    ):
        if not requests or not time_slots:
            raise ValueError("At least one request and one time slot are required.")
        if len({request.id for request in requests}) != len(requests):
            raise ValueError("Request IDs must be unique.")
        if len({slot.id for slot in time_slots}) != len(time_slots):
            raise ValueError("Time-slot IDs must be unique.")
        if any(slot.capacity < 0 for slot in time_slots):
            raise ValueError("Slot capacities must be nonnegative.")
        if num_ants <= 0 or max_iterations <= 0:
            raise ValueError("The ant count and iteration count must be positive.")
        if not 0 <= rho <= 1 or not 0 <= q0 <= 1:
            raise ValueError("rho and q0 must lie between 0 and 1.")
        if alpha < 0 or beta < 0:
            raise ValueError("alpha and beta must be nonnegative.")

        self.requests = requests
        self.time_slots = time_slots
        self.num_requests = len(requests)
        self.num_slots = len(time_slots)
        self.slot_index = {slot.id: index for index, slot in enumerate(time_slots)}

        self.num_ants = num_ants
        self.max_iterations = max_iterations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q0 = q0

        self.tau0 = 1.0 / (self.num_requests * self.num_slots)
        self.pheromone = np.full(
            (self.num_requests, self.num_slots), self.tau0, dtype=float
        )
        self.heuristic = self._calculate_heuristic()

        self.best_solution = None
        self.best_fitness = float("inf")
        self.iteration_best = []
        self.iteration_avg = []
        self.convergence_history = []

        # A fixed seed makes each independently created solver reproducible.
        np.random.seed(42)

    def _calculate_heuristic(self) -> np.ndarray:
        """Score slot choices by their distance from each preferred slot."""
        heuristic = np.zeros((self.num_requests, self.num_slots), dtype=float)
        for request_index, request in enumerate(self.requests):
            for slot_index, slot in enumerate(self.time_slots):
                distance = abs(slot.id - request.preferred_time)
                heuristic[request_index, slot_index] = 1.0 / (1.0 + distance)
        return heuristic

    def _calculate_fitness(self, assignments: Dict[int, int]) -> float:
        """Return the model objective for a complete feasible assignment."""
        if not self._is_feasible(assignments):
            return float("inf")

        total_penalty = 0.0
        for request in self.requests:
            assigned_slot = assignments[request.id]
            deviation = abs(assigned_slot - request.preferred_time)
            total_penalty += request.weight * deviation

        slot_loads = [
            sum(assignments[request.id] == slot.id for request in self.requests)
            for slot in self.time_slots
        ]
        total_penalty += 0.1 * (max(slot_loads) - min(slot_loads))
        return total_penalty

    def _is_feasible(self, assignments: Dict[int, int]) -> bool:
        """Check completeness, valid slot choices, and capacity constraints."""
        if set(assignments) != {request.id for request in self.requests}:
            return False
        if any(slot_id not in self.slot_index for slot_id in assignments.values()):
            return False

        for slot in self.time_slots:
            assigned_count = sum(
                assigned_slot == slot.id for assigned_slot in assignments.values()
            )
            if assigned_count > slot.capacity:
                return False
        return True

    def _construct_solution(self) -> ACOSolution:
        """Build one assignment by sequentially selecting available slots."""
        assignments = {}
        remaining_capacity = {
            slot.id: slot.capacity for slot in self.time_slots
        }
        request_order = list(range(self.num_requests))
        np.random.shuffle(request_order)

        for request_index in request_order:
            request = self.requests[request_index]
            feasible_slot_indices = [
                slot_index
                for slot_index, slot in enumerate(self.time_slots)
                if remaining_capacity[slot.id] > 0
            ]

            if not feasible_slot_indices:
                # An incomplete assignment is retained as infeasible, not repaired.
                continue

            scores = np.array(
                [
                    self.pheromone[request_index, slot_index] ** self.alpha
                    * self.heuristic[request_index, slot_index] ** self.beta
                    for slot_index in feasible_slot_indices
                ],
                dtype=float,
            )
            score_sum = scores.sum()
            if score_sum <= 0 or not np.isfinite(score_sum):
                probabilities = np.full(
                    len(feasible_slot_indices), 1.0 / len(feasible_slot_indices)
                )
            else:
                probabilities = scores / score_sum

            if np.random.random() < self.q0:
                chosen_index = int(np.argmax(probabilities))
            else:
                chosen_index = int(
                    np.random.choice(len(feasible_slot_indices), p=probabilities)
                )

            selected_slot_index = feasible_slot_indices[chosen_index]
            selected_slot_id = self.time_slots[selected_slot_index].id
            assignments[request.id] = selected_slot_id
            remaining_capacity[selected_slot_id] -= 1

        feasible = self._is_feasible(assignments)
        fitness = self._calculate_fitness(assignments) if feasible else float("inf")
        return ACOSolution(assignments, fitness, feasible)

    def _update_pheromone(self, solutions: List[ACOSolution]):
        """Evaporate trails, then reinforce choices in feasible solutions."""
        self.pheromone *= 1.0 - self.rho

        for solution in solutions:
            if solution.feasible:
                # The bounded deposit remains finite for a zero-penalty solution.
                deposit_amount = 1.0 / (1.0 + solution.fitness)
                for request_index, request in enumerate(self.requests):
                    slot_index = self.slot_index[solution.assignments[request.id]]
                    self.pheromone[request_index, slot_index] += deposit_amount

        if self.best_solution is not None and self.best_solution.feasible:
            elite_deposit = 2.0 / (1.0 + self.best_solution.fitness)
            for request_index, request in enumerate(self.requests):
                slot_index = self.slot_index[self.best_solution.assignments[request.id]]
                self.pheromone[request_index, slot_index] += elite_deposit

    def optimize(self) -> Optional[ACOSolution]:
        """Run the ant colony and return its best feasible solution, if found."""
        print(
            f"Starting ACO optimization with {self.num_ants} ants, "
            f"{self.max_iterations} iterations"
        )
        print(
            f"Parameters: alpha={self.alpha}, beta={self.beta}, "
            f"rho={self.rho}, q0={self.q0}"
        )

        for iteration in range(self.max_iterations):
            iteration_solutions = []
            feasible_fitness = []

            for _ in range(self.num_ants):
                solution = self._construct_solution()
                iteration_solutions.append(solution)

                if solution.feasible:
                    feasible_fitness.append(solution.fitness)
                    if solution.fitness < self.best_fitness:
                        self.best_solution = solution
                        self.best_fitness = solution.fitness

            self._update_pheromone(iteration_solutions)

            if feasible_fitness:
                best_in_iteration = min(feasible_fitness)
                average_in_iteration = float(np.mean(feasible_fitness))
                self.iteration_best.append(best_in_iteration)
                self.iteration_avg.append(average_in_iteration)
                if (iteration + 1) % 10 == 0:
                    print(
                        f"Iteration {iteration + 1:3d}: "
                        f"Best={best_in_iteration:.2f}, "
                        f"Average={average_in_iteration:.2f}"
                    )
            else:
                self.iteration_best.append(float("nan"))
                self.iteration_avg.append(float("nan"))

            self.convergence_history.append(
                {
                    "iteration": iteration + 1,
                    "best": self.best_fitness,
                    "average": (
                        float(np.mean(feasible_fitness))
                        if feasible_fitness
                        else float("inf")
                    ),
                    "feasible_solutions": len(feasible_fitness),
                }
            )

        return self.best_solution


print("ACO class defined successfully")

In [ ]:
def run_aco_extended_example():
    """Run ACO on the fixed extended appointment instance."""
    print("=" * 80)
    print("ANT COLONY OPTIMIZATION - EXTENDED EXAMPLE")
    print("=" * 80)

    requests, time_slots = aco_instances["extended"]
    print("\nProblem instance:")
    print(f"  Requests: {len(requests)}")
    print(f"  Time slots: {len(time_slots)}")
    print(f"  Total capacity: {sum(slot.capacity for slot in time_slots)}")

    print("\nRequest details:")
    for request in requests:
        print(
            f"  Request {request.id}: preferred slot {request.preferred_time}, "
            f"weight {request.weight}"
        )

    print("\nTime-slot capacities:")
    for slot in time_slots:
        print(f"  Slot {slot.id}: capacity {slot.capacity}")

    aco = AntColonyOptimization(
        requests,
        time_slots,
        num_ants=20,
        max_iterations=100,
        alpha=1.5,
        beta=2.0,
        rho=0.1,
        q0=0.9,
    )

    start_time = time.time()
    best_solution = aco.optimize()
    execution_time = time.time() - start_time

    print("\n" + "=" * 60)
    print("ACO FINAL RESULTS")
    print("=" * 60)

    if best_solution is None:
        print("No feasible solution was found during this run.")
        return aco, None

    print(f"\nBest penalty: {best_solution.fitness:.2f}")
    print(f"Execution time: {execution_time:.2f} seconds")
    print(f"Solution feasible: {best_solution.feasible}")

    print("\nAssignment details:")
    for request in sorted(requests, key=lambda item: item.id):
        assigned_slot = best_solution.assignments[request.id]
        deviation = abs(assigned_slot - request.preferred_time)
        status = "preferred slot" if deviation == 0 else f"deviation {deviation}"
        print(
            f"  Request {request.id}: slot {assigned_slot} "
            f"(preferred: {request.preferred_time}, weight: {request.weight}; {status})"
        )

    print("\nTime-slot utilization:")
    for slot in time_slots:
        assigned_count = sum(
            best_solution.assignments[request.id] == slot.id for request in requests
        )
        utilization = assigned_count / slot.capacity * 100 if slot.capacity else 0.0
        print(
            f"  Slot {slot.id}: {assigned_count}/{slot.capacity} "
            f"({utilization:.1f}% utilized)"
        )

    total_deviation = sum(
        abs(best_solution.assignments[request.id] - request.preferred_time)
        for request in requests
    )
    weighted_deviation = sum(
        request.weight
        * abs(best_solution.assignments[request.id] - request.preferred_time)
        for request in requests
    )
    requests_at_preferred_time = sum(
        best_solution.assignments[request.id] == request.preferred_time
        for request in requests
    )

    print("\nSolution statistics:")
    print(f"  Total deviation: {total_deviation}")
    print(f"  Weighted deviation: {weighted_deviation:.2f}")
    print(
        f"  Requests at preferred time: "
        f"{requests_at_preferred_time}/{len(requests)}"
    )
    assert aco._is_feasible(best_solution.assignments)
    assert np.isclose(aco._calculate_fitness(best_solution.assignments), best_solution.fitness)
    return aco, best_solution


aco_solver, aco_best_solution = run_aco_extended_example()

In [ ]:
def visualize_aco_performance(aco_solver, best_solution):
    """Plot convergence, learned trails, and the best assignment."""
    if aco_solver is None or best_solution is None:
        print("No feasible ACO solution is available for visualization.")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("Ant Colony Optimization Performance Analysis", fontsize=16)
    requests = aco_solver.requests
    time_slots = aco_solver.time_slots

    ax = axes[0, 0]
    iterations = np.arange(1, len(aco_solver.iteration_best) + 1)
    best_values = np.asarray(aco_solver.iteration_best)
    average_values = np.asarray(aco_solver.iteration_avg)
    valid_best = np.isfinite(best_values)
    valid_average = np.isfinite(average_values)
    ax.plot(iterations[valid_best], best_values[valid_best], label="Iteration best")
    ax.plot(
        iterations[valid_average],
        average_values[valid_average],
        linestyle="--",
        label="Feasible-solution average",
    )
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Fitness (penalty)")
    ax.set_title("Search progress")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    sns.heatmap(
        aco_solver.pheromone.T,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        xticklabels=[f"Request {request.id}" for request in requests],
        yticklabels=[f"Slot {slot.id}" for slot in time_slots],
        ax=ax,
    )
    ax.set_title("Final pheromone trails (larger values indicate stronger trails)")
    ax.set_xlabel("Truck requests")
    ax.set_ylabel("Time slots")

    ax = axes[0, 2]
    assignment_matrix = np.zeros((len(requests), len(time_slots)), dtype=int)
    for request_index, request in enumerate(requests):
        slot_index = aco_solver.slot_index[best_solution.assignments[request.id]]
        assignment_matrix[request_index, slot_index] = 1
    sns.heatmap(
        assignment_matrix,
        annot=True,
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        cbar=False,
        xticklabels=[f"Slot {slot.id}" for slot in time_slots],
        yticklabels=[f"Request {request.id}" for request in requests],
        ax=ax,
    )
    ax.set_title("Best assignment (1 = assigned)")
    ax.set_xlabel("Time slots")
    ax.set_ylabel("Truck requests")

    ax = axes[1, 0]
    sample_request = requests[0]
    request_index = 0
    slot_positions = np.arange(len(time_slots))
    width = 0.35
    ax.bar(
        slot_positions - width / 2,
        aco_solver.pheromone[request_index, :],
        width,
        label="Pheromone",
        color="orange",
    )
    ax.bar(
        slot_positions + width / 2,
        aco_solver.heuristic[request_index, :],
        width,
        label="Heuristic",
        color="blue",
    )
    ax.set_xticks(slot_positions)
    ax.set_xticklabels([f"Slot {slot.id}" for slot in time_slots])
    ax.set_xlabel("Time slots")
    ax.set_ylabel("Value")
    ax.set_title(
        f"Signals for request {sample_request.id} "
        f"(preferred: {sample_request.preferred_time})"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    sampled_fitness = [
        solution.fitness
        for _ in range(100)
        if (solution := aco_solver._construct_solution()).feasible
    ]
    if sampled_fitness:
        ax.hist(
            sampled_fitness,
            bins=20,
            alpha=0.7,
            color="skyblue",
            edgecolor="black",
        )
        ax.axvline(
            best_solution.fitness,
            color="red",
            linestyle="--",
            label=f"Best found: {best_solution.fitness:.2f}",
        )
        ax.set_xlabel("Fitness (penalty)")
        ax.set_ylabel("Frequency")
        ax.set_title("Fitness of 100 additional constructed solutions")
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "No feasible sampled solutions", ha="center", va="center")
        ax.set_title("Additional solution sample")
        ax.set_axis_off()

    ax = axes[1, 2]
    parameter_names = ["alpha", "beta", "rho", "q0"]
    parameter_values = [
        aco_solver.alpha,
        aco_solver.beta,
        aco_solver.rho,
        aco_solver.q0,
    ]
    bars = ax.bar(parameter_names, parameter_values, color=["orange", "blue", "red", "green"])
    ax.set_ylabel("Parameter value")
    ax.set_title("ACO parameter settings")
    ax.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, parameter_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

    finite_iteration_best = best_values[valid_best]
    print("\n" + "=" * 80)
    print("DETAILED ACO ANALYSIS")
    print("=" * 80)
    print("\n1. Convergence analysis:")
    if len(finite_iteration_best) > 0:
        initial_best = finite_iteration_best[0]
        final_best = finite_iteration_best[-1]
        if initial_best > 0:
            improvement = (initial_best - final_best) / initial_best * 100
            print(f"   First feasible iteration best: {initial_best:.2f}")
            print(f"   Last iteration best: {final_best:.2f}")
            print(f"   Change between these values: {improvement:.1f}%")
        else:
            print("   The first feasible iteration already had zero penalty.")
        convergence_point = None
        for index in range(5, len(finite_iteration_best)):
            reference = finite_iteration_best[index - 5]
            if reference > 0:
                recent_improvement = (
                    reference - finite_iteration_best[index]
                ) / reference
                if recent_improvement < 0.01:
                    convergence_point = index + 1
                    break
        if convergence_point is None:
            print("   No five-iteration window met the 1% stopping indicator.")
        else:
            print(
                "   First five-iteration window below the 1% indicator: "
                f"iteration index {convergence_point}"
            )
    else:
        print("   No feasible solution was found in any iteration.")

    print("\n2. Pheromone pattern analysis:")
    flat_indices = np.argsort(aco_solver.pheromone, axis=None)[-5:][::-1]
    print("   Five strongest request-slot trails:")
    for flat_index in flat_indices:
        request_index, slot_index = np.unravel_index(
            flat_index, aco_solver.pheromone.shape
        )
        print(
            f"   Request {requests[request_index].id} -> "
            f"Slot {time_slots[slot_index].id}: "
            f"{aco_solver.pheromone[request_index, slot_index]:.4f}"
        )

    print("\n3. Computational work and storage:")
    print(
        "   Construction and pheromone updates scale approximately as "
        "O(iterations * ants * requests * slots)."
    )
    print("   The pheromone and heuristic matrices each use O(requests * slots) space.")
    print(f"   Iterations run: {len(aco_solver.convergence_history)}")
    print(f"   Ants per iteration: {aco_solver.num_ants}")


visualize_aco_performance(aco_solver, aco_best_solution)

In [ ]:
def parameter_sensitivity_analysis():
    """Examine how ACO parameter settings affect search outcomes."""
    print("\n" + "=" * 80)
    print("ACO PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 80)

    requests, time_slots = aco_instances["extended"]
    parameter_tests = [
        {"alpha": 1.0, "beta": 1.0, "rho": 0.1, "q0": 0.9, "name": "Balanced"},
        {"alpha": 2.0, "beta": 1.0, "rho": 0.1, "q0": 0.9, "name": "Pheromone-heavy"},
        {"alpha": 1.0, "beta": 2.0, "rho": 0.1, "q0": 0.9, "name": "Heuristic-heavy"},
        {"alpha": 1.0, "beta": 1.0, "rho": 0.3, "q0": 0.9, "name": "High evaporation"},
        {"alpha": 1.0, "beta": 1.0, "rho": 0.1, "q0": 0.7, "name": "More exploration"},
        {"alpha": 1.5, "beta": 2.0, "rho": 0.1, "q0": 0.9, "name": "Default"},
    ]

    results = {}
    print("\n1. Parameter-combination tests:")
    print("   Configuration -> best feasible fitness, feasible iterations, runtime")
    for parameters in parameter_tests:
        solver = AntColonyOptimization(
            requests,
            time_slots,
            num_ants=15,
            max_iterations=50,
            alpha=parameters["alpha"],
            beta=parameters["beta"],
            rho=parameters["rho"],
            q0=parameters["q0"],
        )
        start_time = time.time()
        solution = solver.optimize()
        elapsed = time.time() - start_time
        fitness = solution.fitness if solution is not None else float("inf")
        feasible_iterations = sum(
            np.isfinite(value) for value in solver.iteration_best
        )
        results[parameters["name"]] = {
            "fitness": fitness,
            "feasible_iterations": feasible_iterations,
            "time": elapsed,
            "params": parameters,
        }
        fitness_text = f"{fitness:.2f}" if np.isfinite(fitness) else "No feasible solution"
        print(
            f"   {parameters['name']:>18} -> {fitness_text:>20}, "
            f"{feasible_iterations:>3}, {elapsed:>8.2f}s"
        )

    feasible_configurations = [
        name for name, result in results.items() if np.isfinite(result["fitness"])
    ]
    if feasible_configurations:
        best_configuration = min(
            feasible_configurations, key=lambda name: results[name]["fitness"]
        )
        print(f"\nBest tested configuration: {best_configuration}")
        print(f"Best tested fitness: {results[best_configuration]['fitness']:.2f}")
    else:
        print("\nNo tested configuration found a feasible solution.")

    parameter_tests_by_name = {
        "alpha": [0.5, 1.0, 1.5, 2.0, 3.0],
        "beta": [0.5, 1.0, 1.5, 2.0, 3.0],
        "rho": [0.05, 0.1, 0.2, 0.3, 0.5],
    }
    print("\n2. One-parameter-at-a-time tests:")
    for parameter_name, parameter_values in parameter_tests_by_name.items():
        print(f"\n   {parameter_name}:")
        for value in parameter_values:
            settings = {"alpha": 1.5, "beta": 2.0, "rho": 0.1, "q0": 0.9}
            settings[parameter_name] = value
            solver = AntColonyOptimization(
                requests,
                time_slots,
                num_ants=10,
                max_iterations=30,
                **settings,
            )
            solution = solver.optimize()
            fitness = solution.fitness if solution is not None else float("inf")
            fitness_text = f"{fitness:.2f}" if np.isfinite(fitness) else "No feasible solution"
            print(f"     {parameter_name}={value:>4} -> fitness: {fitness_text}")

    return results


sensitivity_results = parameter_sensitivity_analysis()

In [ ]:
class SimpleEDF:
    """Earliest-deadline-first-style baseline used for comparison."""

    def __init__(self, requests, time_slots):
        self.requests = requests
        self.time_slots = time_slots

    def solve(self):
        assignments = {}
        remaining_capacity = {
            slot.id: slot.capacity for slot in self.time_slots
        }
        sorted_requests = sorted(
            self.requests, key=lambda request: request.preferred_time
        )

        for request in sorted_requests:
            preferred_index = next(
                (
                    index
                    for index, slot in enumerate(self.time_slots)
                    if slot.id == request.preferred_time
                ),
                None,
            )
            if preferred_index is None:
                return {"assignments": assignments, "fitness": float("inf"), "feasible": False}

            candidate_indices = [preferred_index]
            for offset in range(1, len(self.time_slots)):
                candidate_indices.extend(
                    [preferred_index + offset, preferred_index - offset]
                )

            assigned = False
            for index in candidate_indices:
                if 0 <= index < len(self.time_slots):
                    slot = self.time_slots[index]
                    if remaining_capacity[slot.id] > 0:
                        assignments[request.id] = slot.id
                        remaining_capacity[slot.id] -= 1
                        assigned = True
                        break
            if not assigned:
                return {"assignments": assignments, "fitness": float("inf"), "feasible": False}

        solver = AntColonyOptimization(
            self.requests,
            self.time_slots,
            num_ants=1,
            max_iterations=1,
        )
        feasible = solver._is_feasible(assignments)
        fitness = solver._calculate_fitness(assignments) if feasible else float("inf")
        return {"assignments": assignments, "fitness": fitness, "feasible": feasible}


def comprehensive_comparison():
    """Compare the included EDF baseline and ACO on the test instances."""
    print("\n" + "=" * 80)
    print("COMPARISON OF EDF AND ACO")
    print("=" * 80)
    comparison_results = {}

    for instance_name, (requests, time_slots) in aco_instances.items():
        print(f"\n--- Instance: {instance_name.upper()} ---")
        print(f"Requests: {len(requests)}, time slots: {len(time_slots)}")
        instance_results = {}

        start_time = time.time()
        edf_solution = SimpleEDF(requests, time_slots).solve()
        edf_time = time.time() - start_time
        instance_results["EDF"] = {
            "fitness": edf_solution["fitness"],
            "time": edf_time,
            "method": "Heuristic",
            "feasible": edf_solution["feasible"],
        }
        edf_fitness_text = (
            f"{edf_solution['fitness']:.2f}"
            if edf_solution["feasible"]
            else "No feasible assignment"
        )
        print(f"  EDF fitness: {edf_fitness_text}; time: {edf_time:.4f}s")

        start_time = time.time()
        aco_solver = AntColonyOptimization(
            requests,
            time_slots,
            num_ants=min(20, len(requests)),
            max_iterations=min(50, len(requests) * 2),
            alpha=1.5,
            beta=2.0,
            rho=0.1,
            q0=0.9,
        )
        aco_solution = aco_solver.optimize()
        aco_time = time.time() - start_time
        aco_fitness = aco_solution.fitness if aco_solution is not None else float("inf")
        instance_results["ACO"] = {
            "fitness": aco_fitness,
            "time": aco_time,
            "method": "Metaheuristic",
            "feasible": aco_solution is not None,
        }
        aco_fitness_text = (
            f"{aco_fitness:.2f}" if aco_solution is not None else "No feasible assignment"
        )
        print(f"  ACO fitness: {aco_fitness_text}; time: {aco_time:.4f}s")

        if (
            aco_solution is not None
            and edf_solution["feasible"]
            and edf_solution["fitness"] > 0
        ):
            improvement = (
                (edf_solution["fitness"] - aco_fitness)
                / edf_solution["fitness"]
                * 100
            )
            print(f"  ACO change relative to EDF fitness: {improvement:+.1f}%")
        comparison_results[instance_name] = instance_results

    print("\nSummary comparison:")
    print("Instance          | EDF fitness | ACO fitness | ACO change")
    print("-" * 66)
    for instance_name, results in comparison_results.items():
        edf_result = results["EDF"]
        aco_result = results["ACO"]
        edf_text = f"{edf_result['fitness']:.2f}" if edf_result["feasible"] else "infeasible"
        aco_text = f"{aco_result['fitness']:.2f}" if aco_result["feasible"] else "not found"
        if (
            edf_result["feasible"]
            and aco_result["feasible"]
            and edf_result["fitness"] > 0
        ):
            change_text = (
                f"{(edf_result['fitness'] - aco_result['fitness']) / edf_result['fitness'] * 100:+.1f}%"
            )
        else:
            change_text = "N/A"
        print(f"{instance_name:>16} | {edf_text:>11} | {aco_text:>11} | {change_text:>10}")

    print("\nScalability timing experiment:")
    print("Requests | EDF time (ms) | ACO time (ms) | ACO/EDF time")
    print("-" * 59)
    for size in [6, 12, 18, 24, 30]:
        number_of_slots = min(8, size // 3)
        test_requests = [
            TruckRequest(
                request_id,
                np.random.randint(0, number_of_slots),
                np.random.uniform(0.5, 2.0),
            )
            for request_id in range(1, size + 1)
        ]
        test_slots = [
            TimeSlot(slot_id, max(2, size // 6))
            for slot_id in range(number_of_slots)
        ]

        start_time = time.time()
        SimpleEDF(test_requests, test_slots).solve()
        edf_time_ms = (time.time() - start_time) * 1000

        start_time = time.time()
        AntColonyOptimization(
            test_requests,
            test_slots,
            num_ants=min(10, size),
            max_iterations=min(20, size),
            alpha=1.5,
            beta=2.0,
            rho=0.1,
            q0=0.9,
        ).optimize()
        aco_time_ms = (time.time() - start_time) * 1000
        time_ratio = aco_time_ms / edf_time_ms if edf_time_ms > 0 else float("inf")
        print(f"{size:>8} | {edf_time_ms:>13.2f} | {aco_time_ms:>13.2f} | {time_ratio:>12.1f}x")

    return comparison_results


comparison_results = comprehensive_comparison()

## Strengths, limitations, and appropriate use

ACO provides a structured way to search for good assignments when a simple construction heuristic may not find a low-penalty solution. Its pheromone trails retain information from previous solutions, and its stochastic choices allow exploration as well as repeated use of promising choices. The included EDF comparison provides a baseline; it does not imply that ACO will outperform EDF on every instance.

ACO does not guarantee a globally optimal assignment. Its performance depends on the instance, parameter settings, and random choices, and different seeds can lead to different search paths. The sensitivity experiment varies parameters while holding the extended instance fixed; it is an empirical comparison, not a general proof of which settings are best. Runtime measurements are also specific to the execution environment.

The construction and pheromone updates require work proportional to the number of ants, iterations, requests, and slots, approximately \(O(T\,m\,|R|\,|S|)\), where \(T\) is the iteration count and \(m\) is the number of ants. The pheromone and heuristic matrices each require \(O(|R|\,|S|)\) storage. These expressions describe the dominant operations in this implementation; they do not promise a particular runtime or solution quality.

ACO is useful when a flexible search for good feasible assignments is valuable and an optimality certificate is not required. Additional operational constraints can be incorporated only when the solution-construction and feasibility logic are also defined to handle them.